In [14]:
import anthropic
import pandas as pd
import re
import time
from pathlib import Path
import os

In [ ]:
API_KEY = ""
MODEL = "claude-opus-4-5"
DELAY_BETWEEN_CALLS = 1.0

In [10]:
def load_speech(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Speech file not found: {filepath}")
    if path.suffix.lower() != ".txt":
        raise ValueError(f"Expected a .txt file, got: {path.suffix}")
    
    raw = path.read_text(encoding="utf-8", errors="replace")
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    test = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_training_set(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Trianing set file not found: {filepath}")
    
    df = pd.read_excel(path)
    df.columns = df.columns.str.strip()
    df["Text"] = df["Text"].astype(str).str.strip()
    df = df[df["Text"].notna() & (df["Text"] != "") & (df["Text"] != "nan")]
    df = df.reset_index(drop=True)

    score_columns = [col for col in df.columns if col != "Text"]
    for col in score_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df





In [11]:
def build_prompt(text, training_df):
    few_shot = ""
    for _, row in training_df.iterrows():
        few_shot += f'Text: "{row["Text"]}"\nScore: {int(row["Adopt us vs them framing"])}\n\n'

    prompt = f"""You are a political language analyst. Below are examples of texts scored for "us vs them" framing intensity.

{few_shot}---

Now score the following text using this rubric:

Count all instances of ingroup references, which include defined collective identities such as "we," "us," "our," or named groups like "Americans," "working people," or "tenants." Do not count generic terms like "you," "everyone," or "people" unless they clearly refer to a specific, defined group. Count all instances of outgroup references, including words like "they" or "them," as well as named opposing groups or actors such as "Republicans," "Democrats," "landlords," or "the other side" if they are being referred to as an opposing group. Add one point for each sentence that contains a moral attack, meaning language that portrays the outgroup as corrupt, selfish, dishonest, hypocritical, or unfair. Add one point for each sentence that contains threat or harm framing, meaning language that describes negative consequences, danger, or harm, such as people "falling behind," a system being "rigged," or outcomes like rising costs or exclusion. Add one point if there is a call for unity against the outgroup, such as language urging people to stand together, fight back, or stop them.

Before giving the final score, list the counts for each category in the following format:
Ingroup: X, Outgroup: X, Moral: X, Threat: X, Unity: X

Then provide the final score as the sum of these values. Finally, output the result as:
Score: X

Text to score:
\"\"\"{text}\"\"\"
"""
    return prompt


def call_api(text, training_df):
    client = anthropic.Anthropic(api_key=API_KEY)
    prompt = build_prompt(text, training_df)

    message = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

In [ ]:
def parse_response(raw):
    counts_pattern = re.search(
        r"Ingroup:\s*(\d+),\s*Outgroup:\s*(\d+),\s*Moral:\s*(\d+),\s*Threat:\s*(\d+),\s*Unity:\s*(\d+)",
        raw, re.IGNORECASE
    )
    score_pattern = re.search(r"Score:\s*(\d+)", raw, re.IGNORECASE)

    result = {
        "ingroup":  int(counts_pattern.group(1)) if counts_pattern else None,
        "outgroup": int(counts_pattern.group(2)) if counts_pattern else None,
        "moral":    int(counts_pattern.group(3)) if counts_pattern else None,
        "threat":   int(counts_pattern.group(4)) if counts_pattern else None,
        "unity":    int(counts_pattern.group(5)) if counts_pattern else None,
        "score":    int(score_pattern.group(1))  if score_pattern  else None,
        "raw_response": raw,
        "parse_error": None
    }

    if not counts_pattern:
        result["parse_error"] = "Could not parse category counts"
    if not score_pattern:
        result["parse_error"] = (result["parse_error"] or "") + "; Could not parse score"

    return result



In [13]:
def score_speech(speech_path, training_path="training_set.xlsx"):
    speech = load_speech(speech_path)
    training_df = load_training_set(training_path)

    raw = call_api(speech, training_df)
    result = parse_response(raw)

    result["filename"] = Path(speech_path).name

    return result


result = score_speech(
    speech_path="Zara-Ahsan-Thesis-Repository/speeches/andy beshear/andy_beshear_final_debate.txt",
)

print(result)

{'ingroup': 50, 'outgroup': 55, 'moral': 18, 'threat': 15, 'unity': 3, 'score': 114, 'raw_response': 'I\'ll analyze this text systematically for "us vs them" framing intensity.\n\n**Ingroup references:**\n- "we" (appears frequently throughout - approximately 45+ instances)\n- "our" (community colleges, pension system, teachers, commonwealth, kids, families, etc. - approximately 30+ instances)\n- "us" (several instances)\n- "Kentuckians" (multiple instances)\n- "teachers" (as defended group)\n- "police officers," "firefighters," "social workers"\n- "families," "students"\n- References to "my running mate," "my ticket"\n\n**Outgroup references:**\n- "Matt Bevin" / "this governor" / "my opponent" / "he" (referring to opponent - approximately 50+ instances)\n- "these companies" (pharma)\n- "landlords and developers" (not present, wrong text)\n- References to opponent\'s actions and allies\n\n**Counting systematically:**\n\n**Ingroup references:** ~50 (we, our, us, Kentuckians, teachers, fa

In [17]:
def score_all_speeches(speech_folders: list, training_path: str, output_path: str):
    results = []

    for folder in speech_folders:
        for filename in os.listdir(folder):
            if filename.endswith(".txt"):
                speech_path = os.path.join(folder, filename)
                print(f"Scoring: {speech_path}")
                result = score_speech(speech_path, training_path)
                results.append(result)
                time.sleep(DELAY_BETWEEN_CALLS)

    df = pd.DataFrame(results)
    df.to_excel(output_path, index=False)
    print(f"\nDone! Results saved to {output_path}")
    return df


folders = [
    "Zara-Ahsan-Thesis-Repository/speeches/andy beshear",
    "Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown",
    "Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker",
    "Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach",
    "Zara-Ahsan-Thesis-Repository/speeches/Larry Hogan",
    "Zara-Ahsan-Thesis-Repository/speeches/Laura Kelly",
    "Zara-Ahsan-Thesis-Repository/speeches/Martha Coakley",
    "Zara-Ahsan-Thesis-Repository/speeches/matt bevin",
    "Zara-Ahsan-Thesis-Repository/speeches/Phil Scott",
    "Zara-Ahsan-Thesis-Repository/speeches/Sue Minter"
]

df_results = score_all_speeches(
    speech_folders=folders,
    training_path="training_set.xlsx",
    output_path="scored_speeches.xlsx"
)

Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_final_debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_partisan_winning_speech.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_tweets.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\Fancy Farms.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown\Fox interview.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown\Oct 7 guber debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\MACDC's 2014 Convention.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\Oct 21 Gubernatorial debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\Oct 23, 2014 Massachusetts Gubernatorial Debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\The Berkshire Eagle.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach\2018 Kansas Governor Debate - September 5, 2018